# Notebook de definición del problema y entorno reproducible — Fase 1

**Proyecto transversal · MCDI500 Programación para la Ciencia de Datos**
**Magíster en Ciencia de Datos e Inteligencia Artificial · Universidad Andrés Bello**

**Grupo 5 — Factores asociados a la duración autorizada de los permisos de uso de vía
pública en San Francisco**

## Introducción

Este cuaderno deja establecido lo que la Fase 2 necesita para empezar: la definición del
problema, un entorno verificado, la estructura del repositorio con sus artefactos de
reproducibilidad, un módulo propio importable y la documentación del conjunto de datos con su
evaluación contra los criterios del curso.

No transforma datos: los define. La limpieza, la imputación y el escalamiento son trabajo de
la Fase 2, que lee lo que aquí se decide en lugar de volver a decidirlo.

**Orden de ejecución.** Lineal, de arriba abajo. *Kernel → Restart Kernel and Run All Cells*
debe terminar sin errores y con la numeración de ejecución continua.

### Herramientas del ecosistema científico

Cada importación responde a una necesidad concreta de esta fase y no se incluye ninguna que
no se use. `pathlib` resuelve rutas de forma independiente del sistema operativo; `subprocess`
y `shutil` consultan el estado real de Git sin suponer que esté instalado; `json` exporta los
metadatos en un formato legible tanto por una persona como por el cuaderno de la Fase 2.

In [1]:
import sys                       # intérprete en uso: es lo que se verifica más abajo
import json                      # exportación de metadatos legibles
import platform                  # sistema operativo, para dejarlo en la bitácora
import subprocess                # consultas a Git desde el cuaderno
import shutil                    # localizar ejecutables (git) sin suponer que existen
import importlib                 # importar el módulo que este cuaderno va a escribir
import warnings                  # capturar los avisos de parseo como evidencia, no como ruido
import io, contextlib            # capturar salidas para los anexos del informe
from pathlib import Path         # manejo de rutas independiente del sistema operativo
from datetime import date

import numpy as np               # presente para verificar el entorno del proyecto
import pandas as pd              # tablas de documentación de la fase

# Reproducibilidad: la misma semilla que usa el cuaderno de la Fase 2.
SEMILLA = 42
np.random.seed(SEMILLA)

def capturar(funcion, *args, **kwargs):
    """
    Ejecuta una funcion, muestra su salida en pantalla y ademas la devuelve como texto.

    Sirve para que los anexos del informe se generen desde la misma ejecucion que produce
    los resultados, en lugar de copiarse a mano desde una captura de pantalla.

    Retorna
    -------
    (obj, str)
        Lo que devuelva la funcion y su salida estandar como texto.
    """
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        resultado = funcion(*args, **kwargs)
    texto = buffer.getvalue()
    print(texto, end="")
    return resultado, texto


print("Cuaderno de la Fase 1 ·", date.today().isoformat())

Cuaderno de la Fase 1 · 2026-09-15


## 1. Definición del problema

Toda la identificación del proyecto vive en un único diccionario. Lo que el cuaderno imprima
o persista después se deriva de aquí, de modo que no circulen dos versiones del título, de la
pregunta o de los integrantes.

> **Completar antes de entregar:** los nombres de los cuatro integrantes. El correo
> configurado en Git debe coincidir con el de su cuenta de GitHub, o los *commits* no se les
> atribuyen.

In [2]:
PROYECTO = {
    "titulo": "Factores asociados a la duración autorizada de los permisos de uso de vía "
              "pública en San Francisco",
    "grupo": "Grupo 5",
    "asignatura": "MCDI500 · Programación para la Ciencia de Datos",
    "integrantes": [
        "MARCO ÁLVAREZ ARAYA (@Marco30101994-Unab)",
        "MARÍA FASSLER NEUMANN (@belenfassler)",
        "RICARDO JARAMILLO PULGAR (@RJaramilloP)"],
    "problematica": (
        "Los permisos de uso de vía pública regulan la ocupación temporal del espacio público "
        "por excavaciones, obras, instalaciones de telecomunicaciones y usos comerciales. La "
        "duración autorizada de cada permiso determina cuánto tiempo una calle queda afectada. "
        "El problema es de asociación: identificar qué factores se relacionan con esa duración, "
        "medida como el número de días entre la fecha de inicio y la de término del permiso."
    ),
    "objetivo_general": (
        "Identificar y cuantificar los factores asociados a la duración autorizada de los "
        "permisos de uso de vía pública vigentes en San Francisco, mediante un flujo "
        "reproducible de obtención, saneamiento y validación de datos abiertos."
    ),
    "objetivos_especificos": [
        "Definir el problema y establecer el entorno reproducible del proyecto (F1).",
        "Documentar la procedencia, la licencia y el rol analítico de cada variable (F1).",
        "Construir el pipeline de obtención, limpieza y transformación de datos (F2).",
        "Implementar el núcleo algorítmico con programación estructurada y POO (F3).",
        "Comunicar los hallazgos mediante visualización y un informe técnico (F4).",
    ],
    "preguntas": [
        "¿Qué proporción de la variación en la duración se asocia al tipo de permiso?",
        "Dentro de un mismo tipo, ¿el barrio y el agente solicitante distinguen duraciones?",
        "¿Qué variables concentran los faltantes y qué significa que un dato no esté?",
    ],
    "criterios_exito": [
        "El repositorio se clona y ambos cuadernos corren completos sin intervención manual.",
        "Cada decisión de preprocesamiento queda justificada con cifras, no con adjetivos.",
        "Los cuatro integrantes tienen commits propios en el historial.",
    ],
    "alcance": {
        "incluye": ["Definición del problema", "Entorno reproducible",
                    "Selección y documentación del conjunto"],
        "excluye": ["Limpieza e imputación", "Modelado", "Inferencia causal"],
    },
    "limitaciones": [
        "Los datos son secundarios: no se controló su recolección ni su representatividad.",
        "La fuente publica solo permisos vigentes, de modo que los de mayor duración quedan "
        "sobrerrepresentados. Se cuantifica en la sección 5.4.",
        "El portal se actualiza a diario: los resultados corresponden a un corte concreto y no "
        "son reproducibles descargando el archivo de nuevo.",
    ],
}

La función siguiente presenta el proyecto en pantalla. Recibe el diccionario como
parámetro en lugar de leer una variable global, de modo que pueda reutilizarse y probarse con
cualquier configuración. Falla temprano y con un mensaje que dice **qué** falta, no solo que
algo salió mal.

In [3]:
def presentar_proyecto(config):
    """
    Imprime la definicion del proyecto de forma legible.

    Parametros
    ----------
    config : dict
        Diccionario de configuracion del proyecto.

    Retorna
    -------
    None

    Lanza
    -----
    KeyError
        Si falta alguna de las claves obligatorias.
    """
    obligatorias = ("titulo", "problematica", "objetivo_general", "objetivos_especificos")
    faltantes = [c for c in obligatorias if c not in config]
    if faltantes:
        # Se falla temprano y con un mensaje que dice QUE falta, no solo que algo fallo.
        raise KeyError(f"Faltan claves obligatorias en la configuracion: {faltantes}")

    print(config["titulo"].upper())
    print("=" * 60)
    print(f"{config['asignatura']} · {config['grupo']}")
    print("\nProblemática")
    print(config["problematica"])
    print("\nObjetivo general")
    print(config["objetivo_general"])
    print("\nObjetivos específicos")
    for i, obj in enumerate(config["objetivos_especificos"], start=1):
        print(f"  {i}. {obj}")
    print("\nPreguntas que orientan el trabajo")
    for pregunta in config["preguntas"]:
        print(f"  · {pregunta}")


presentar_proyecto(PROYECTO)

FACTORES ASOCIADOS A LA DURACIÓN AUTORIZADA DE LOS PERMISOS DE USO DE VÍA PÚBLICA EN SAN FRANCISCO
MCDI500 · Programación para la Ciencia de Datos · Grupo 5

Problemática
Los permisos de uso de vía pública regulan la ocupación temporal del espacio público por excavaciones, obras, instalaciones de telecomunicaciones y usos comerciales. La duración autorizada de cada permiso determina cuánto tiempo una calle queda afectada. El problema es de asociación: identificar qué factores se relacionan con esa duración, medida como el número de días entre la fecha de inicio y la de término del permiso.

Objetivo general
Identificar y cuantificar los factores asociados a la duración autorizada de los permisos de uso de vía pública vigentes en San Francisco, mediante un flujo reproducible de obtención, saneamiento y validación de datos abiertos.

Objetivos específicos
  1. Definir el problema y establecer el entorno reproducible del proyecto (F1).
  2. Documentar la procedencia, la licencia y el rol 

El alcance y las limitaciones se presentan aparte porque cumplen una función distinta:
el primero delimita qué entra en esta entrega y qué no; las segundas declaran qué no puede
afirmarse con estos datos, por bien que se haga el trabajo.

In [4]:
print("Alcance de la Fase 1")
print("  Incluye:", ", ".join(PROYECTO["alcance"]["incluye"]))
print("  Excluye:", ", ".join(PROYECTO["alcance"]["excluye"]))

print("\nLimitaciones declaradas")
for lim in PROYECTO["limitaciones"]:
    print(f"  · {lim}")

Alcance de la Fase 1
  Incluye: Definición del problema, Entorno reproducible, Selección y documentación del conjunto
  Excluye: Limpieza e imputación, Modelado, Inferencia causal

Limitaciones declaradas
  · Los datos son secundarios: no se controló su recolección ni su representatividad.
  · La fuente publica solo permisos vigentes, de modo que los de mayor duración quedan sobrerrepresentados. Se cuantifica en la sección 5.4.
  · El portal se actualiza a diario: los resultados corresponden a un corte concreto y no son reproducibles descargando el archivo de nuevo.


> **Sobre las limitaciones.** Declararlas antes de haber analizado nada es señal de haber
> entendido la fuente, no de desconfianza en el trabajo. La segunda de ellas —la
> sobrerrepresentación de los permisos largos— no es una sospecha: la sección 5.4 la mide y
> muestra que la duración mínima observada crece con la antigüedad del permiso, que es
> exactamente lo que predice el mecanismo.

## 2. Verificación del entorno

Se registra el entorno **realmente en uso**, no el que se esperaba tener. Si alguna versión
difiere entre integrantes, esta salida lo revela antes de que se convierta en un resultado
distinto sin explicación.

In [5]:
DEPENDENCIAS = ["numpy", "pandas", "matplotlib", "sklearn"]

# Correspondencia entre el nombre de importación y el nombre del paquete en PyPI:
# se instala scikit-learn, pero se importa sklearn.
NOMBRE_EN_PYPI = {"sklearn": "scikit-learn"}


def verificar_entorno(dependencias):
    """
    Comprueba interprete, entorno virtual, carpeta de trabajo y librerias.

    Parametros
    ----------
    dependencias : list of str
        Nombres de importacion de las librerias que el proyecto declara.

    Retorna
    -------
    dict
        Resultado de cada comprobacion, para dejarlo en la bitacora de la fase.
    """
    reporte = {}

    # 1. Interprete que ejecuta ESTE cuaderno. Si la ruta no contiene .venv,
    #    el kernel no es el del proyecto: Kernel -> Change Kernel.
    ejecutable = Path(sys.executable)
    en_venv = ".venv" in ejecutable.parts or sys.prefix != sys.base_prefix
    reporte["interprete"] = str(ejecutable)
    reporte["entorno_virtual"] = bool(en_venv)
    print("Intérprete       :", ejecutable)
    print("Entorno virtual  :", "[OK] activo" if en_venv else "[AVISO] parece el Python del sistema")

    # 2. Carpeta de trabajo: las rutas relativas se resuelven desde aqui,
    #    no desde donde esta guardado el archivo .ipynb.
    reporte["carpeta_trabajo"] = str(Path.cwd())
    print("Carpeta de trabajo:", Path.cwd())
    print("Sistema          :", platform.system(), platform.release())
    print("Python           :", sys.version.split()[0])

    # 3. Librerias del proyecto, con su version.
    print("\nLibrerías declaradas")
    versiones = {}
    for nombre in dependencias:
        try:
            modulo = importlib.import_module(nombre)
            version = getattr(modulo, "__version__", "sin atributo __version__")
            versiones[nombre] = version
            print(f"  [OK]    {nombre:12} {version}")
        except ImportError:
            # No se interrumpe el cuaderno: se informa que instalar y con que nombre.
            versiones[nombre] = None
            paquete = NOMBRE_EN_PYPI.get(nombre, nombre)
            print(f"  [FALTA] {nombre:12} instale con: python -m pip install {paquete}")
    reporte["versiones"] = versiones
    return reporte


ENTORNO, ANEXO_A = capturar(verificar_entorno, DEPENDENCIAS)   # Anexo A del informe

Intérprete       : /opt/anaconda3/bin/python
Entorno virtual  : [AVISO] parece el Python del sistema
Carpeta de trabajo: /Users/ricardo/Desktop/Repositorios/proyecto-grupo5-mcdi500/F1/notebooks
Sistema          : Darwin 25.6.0
Python           : 3.9.13

Librerías declaradas
  [OK]    numpy        1.26.4
  [OK]    pandas       2.3.3
  [OK]    matplotlib   3.5.2
  [OK]    sklearn      1.0.2


**Cómo se lee este resultado.** Cada línea corresponde a un acoplamiento que la
reproducibilidad exige explicitar. Si el intérprete no apunta al `.venv` del proyecto, el
cuaderno está corriendo con las librerías del sistema y las versiones que se registren más
abajo no serán las que el grupo declaró. Si alguna librería aparece como `[FALTA]`, el
cuaderno no se interrumpe: informa qué instalar, porque detenerse obligaría a reejecutar todo
desde el principio por un problema que se resuelve en una línea.

## 3. Estructura del repositorio y artefactos

Las rutas de datos son **comunes a todas las fases**: son exactamente las que usa el cuaderno
de la Fase 2, de modo que un mismo archivo sirva a los dos. Cada fase tiene su propia carpeta
para cuadernos e informes.

In [6]:
MARCADORES = (".git", "requirements.txt", ".gitignore")


def localizar_raiz(inicio=None, marcadores=MARCADORES, niveles=5, respaldo=True):
    """
    Resuelve la raiz del proyecto subiendo hasta encontrar un marcador.

    Parametros
    ----------
    inicio : str | Path | None
        Punto de partida. Por defecto, el directorio de trabajo actual.
    marcadores : tuple of str
        Archivos o carpetas que identifican la raiz de un repositorio.
    niveles : int
        Numero maximo de directorios padre a recorrer.
    respaldo : bool
        Si es True y no hay marcador, retorna `inicio` (disposicion plana).

    Retorna
    -------
    (Path, str)
        Ruta de la raiz y modo detectado: "estructurado" o "plano".

    Lanza
    -----
    FileNotFoundError
        Si no hay marcador y respaldo es False.
    """
    actual = Path(inicio or Path.cwd()).resolve()
    for candidata in [actual, *actual.parents][:niveles + 1]:
        if any((candidata / m).exists() for m in marcadores):
            return candidata, "estructurado"
    if respaldo:
        return actual, "plano"
    raise FileNotFoundError(f"No se encontro marcador de repositorio desde {actual}.")


RAIZ, MODO = localizar_raiz()        # raiz del proyecto, no del directorio de lanzamiento

# Rutas de datos COMUNES a todas las fases.
DIR_CRUDO = RAIZ / "data" / "raw"            # datos originales: nunca se modifican
DIR_PROCESADO = RAIZ / "data" / "processed"  # salida del pipeline de la Fase 2
DIR_DOCS = RAIZ / "docs"                     # diccionario, fichas y metadatos
DIR_SRC = RAIZ / "src"                       # modulos propios reutilizables

# Una carpeta por fase para cuadernos e informes.
DIR_FASES = [RAIZ / f"F{n}" for n in (1, 2, 3, 4)]

for carpeta in [DIR_CRUDO, DIR_PROCESADO, DIR_DOCS, DIR_SRC, *DIR_FASES]:
    # parents=True crea las intermedias; exist_ok=True evita el error si ya existen,
    # que es lo que permite volver a ejecutar el cuaderno completo sin romperlo.
    carpeta.mkdir(parents=True, exist_ok=True)

print("Raíz del proyecto :", RAIZ)
print("Modo detectado    :", MODO)
print("Ejecutado desde   :", Path.cwd())
print("\nEstructura creada:")
for ruta in sorted(p for p in RAIZ.rglob("*") if p.is_dir() and ".git" not in p.parts
                   and ".ipynb_checkpoints" not in p.parts and "__pycache__" not in p.parts):
    print("  ", ruta.relative_to(RAIZ).as_posix() + "/")

Raíz del proyecto : /Users/ricardo/Desktop/Repositorios/proyecto-grupo5-mcdi500
Modo detectado    : estructurado
Ejecutado desde   : /Users/ricardo/Desktop/Repositorios/proyecto-grupo5-mcdi500/F1/notebooks

Estructura creada:
   F1/
   F1/.venv/
   F1/.venv/bin/
   F1/.venv/include/
   F1/.venv/lib/
   F1/.venv/lib/python3.9/
   F1/.venv/lib/python3.9/site-packages/
   F1/.venv/lib/python3.9/site-packages/_distutils_hack/
   F1/.venv/lib/python3.9/site-packages/pip/
   F1/.venv/lib/python3.9/site-packages/pip/_internal/
   F1/.venv/lib/python3.9/site-packages/pip/_internal/cli/
   F1/.venv/lib/python3.9/site-packages/pip/_internal/commands/
   F1/.venv/lib/python3.9/site-packages/pip/_internal/distributions/
   F1/.venv/lib/python3.9/site-packages/pip/_internal/index/
   F1/.venv/lib/python3.9/site-packages/pip/_internal/locations/
   F1/.venv/lib/python3.9/site-packages/pip/_internal/metadata/
   F1/.venv/lib/python3.9/site-packages/pip/_internal/models/
   F1/.venv/lib/python3.9/site

> **Decisión que el grupo tomó y documenta.** La guía propone fijar la raíz como
> `Path(".")`, es decir, la carpeta desde la que se lanza el cuaderno. Como este cuaderno vive
> en `F1/notebooks/`, esa expresión crearía `data/`, `docs/` y `src/` **dentro** de
> `F1/notebooks/`, y el cuaderno de la Fase 2 —que se lanza desde `F2/`— apuntaría a un árbol
> distinto. El resultado sería el mismo proyecto con dos estructuras paralelas y datos
> duplicados.
>
> Se sustituye por `localizar_raiz()`, que sube hasta encontrar un marcador de repositorio. El
> cuaderno queda invariante ante el directorio de lanzamiento, que es la condición para que un
> tercero pueda reproducirlo.
>
> *Enfoques descartados:* una ruta absoluta, que funciona en un solo computador; y `os.chdir()`,
> que deja el intérprete en un estado distinto según cuántas veces se ejecute la celda.

### 3.1 El archivo `.gitignore`

Define qué queda fuera del control de versiones. Se escribe desde el cuaderno y se relee para
comprobar que quedó como se esperaba: escribir un archivo y darlo por hecho es la clase de
suposición que después cuesta horas encontrar.

In [7]:
CONTENIDO_GITIGNORE = """# Entorno virtual: se reconstruye con requirements.txt
.venv/
venv/

# Caché de Python
__pycache__/
*.pyc

# Jupyter
.ipynb_checkpoints/

# Windows
Thumbs.db
desktop.ini

# macOS
.DS_Store

# Variables de entorno y secretos
.env

# Datos pesados y modelos
*.ckpt
*.pth
"""

ARCHIVO_GITIGNORE = RAIZ / ".gitignore"
ARCHIVO_GITIGNORE.write_text(CONTENIDO_GITIGNORE, encoding="utf-8")

# Verificacion de ida y vuelta: se relee lo escrito antes de darlo por hecho.
lineas = [l for l in ARCHIVO_GITIGNORE.read_text(encoding="utf-8").splitlines()
          if l and not l.startswith("#")]
assert ".venv/" in lineas, "El entorno virtual debe estar ignorado."
assert ".ipynb_checkpoints/" in lineas, "Los checkpoints de Jupyter no deben versionarse."
print(f"{ARCHIVO_GITIGNORE.relative_to(RAIZ).as_posix()} escrito con {len(lineas)} reglas activas.")

.gitignore escrito con 11 reglas activas.


> **Atención — los datos.** Este `.gitignore` **no** ignora `data/`, y es una decisión
> deliberada. La fuente de este proyecto se actualiza a diario: volver a descargar el archivo
> produce un conjunto distinto. El archivo versionado en `data/raw/` **es** el dato, no una
> copia de conveniencia, y sin él los resultados no se pueden reproducir. Pesa 3,4 MB, lo que
> está muy por debajo de cualquier límite razonable para un repositorio.

### 3.2 El archivo `requirements.txt`

Se genera a partir de las versiones **realmente presentes** en el entorno, no de una lista
escrita a mano. Un archivo de dependencias que no corresponde al entorno que produjo los
resultados no es reproducibilidad: es documentación inexacta.

In [8]:
def escribir_requirements(ruta, versiones):
    """
    Escribe requirements.txt a partir de las versiones observadas en el entorno.

    Parametros
    ----------
    ruta : Path
        Archivo de salida.
    versiones : dict
        Nombre de importacion -> version detectada (o None si falta).

    Retorna
    -------
    list of str
        Lineas escritas, para verificarlas.

    Lanza
    -----
    ValueError
        Si ninguna dependencia pudo detectarse.
    """
    lineas = []
    for nombre, version in versiones.items():
        if version is None:
            continue  # una libreria ausente no se declara: primero se instala
        paquete = NOMBRE_EN_PYPI.get(nombre, nombre)
        lineas.append(f"{paquete}=={version}")

    if not lineas:
        raise ValueError("No se detecto ninguna dependencia instalada.")

    # Se agregan las herramientas del flujo de trabajo, que no se importan desde el
    # codigo pero si forman parte del entorno reproducible.
    lineas += ["jupyterlab", "notebook", "ipykernel"]
    ruta.write_text("\n".join(sorted(lineas)) + "\n", encoding="utf-8")
    return sorted(lineas)


ARCHIVO_REQUIREMENTS = RAIZ / "requirements.txt"
requisitos = escribir_requirements(ARCHIVO_REQUIREMENTS, ENTORNO["versiones"])
print(ARCHIVO_REQUIREMENTS.relative_to(RAIZ).as_posix(), "\n")
print("\n".join(requisitos))

requirements.txt 

ipykernel
jupyterlab
matplotlib==3.5.2
notebook
numpy==1.26.4
pandas==2.3.3
scikit-learn==1.0.2


> **Atención.** Este archivo se generó desde las librerías presentes en el entorno de quien
> ejecutó el cuaderno. Si un integrante lo ejecuta con versiones distintas, `requirements.txt`
> cambiará y el cambio aparecerá en el `git diff`. Eso no es un defecto: es precisamente la
> señal de que los entornos divergieron, y conviene resolverla antes de comparar resultados.

## 4. Módulos: del cuaderno a `src/`

Las funciones que ambos cuadernos necesitan se escriben una sola vez, en `src/utilidades.py`,
y se importan. Duplicarlas entre la Fase 1 y la Fase 2 garantiza que en algún momento
diverjan.

El módulo incluye `construir_conjunto_trabajo()`, que implementa en una sola función el plan
de saneamiento justificado en la sección 5.2. El cuaderno de la Fase 2 reconstruye ese
conjunto paso a paso y a la vista, y al final contrasta su resultado con el de esta función:
dos caminos independientes que deben coincidir.

In [9]:
CODIGO_MODULO = '''"""Utilidades compartidas del proyecto MCDI500 - Grupo 5.

Este modulo lo genera el cuaderno de la Fase 1 y lo importan las fases siguientes.
Conjunto: permisos de uso de via publica de San Francisco (DataSF, x8nh-xzn6).
"""

import re
from pathlib import Path

import numpy as np
import pandas as pd

MARCADORES = (".git", "requirements.txt", ".gitignore")

FORMATO_FECHA = "%m/%d/%Y"
FORMATO_SELLO = "%Y/%m/%d %I:%M:%S %p"
COLS_FECHA = ["Approved Date", "permit_start_date", "permit_end_date"]
COLS_COORDENADA = ["Latitude", "Longitude"]

# Columnas descartadas y su motivo. Ver seccion 5.2 del cuaderno de la Fase 1.
DESCARTES = {
    "CurbRampWork": "96,4% de nulos y un unico valor distinto: varianza cero",
    "bpa": "95,7% de nulos",
    "permit_address": "63,1% de nulos; redundante con streetname y cross streets",
    "Location": "geometria en texto, redundante con Latitude y Longitude",
    "the_geom": "geometria en texto, redundante con Latitude y Longitude",
    "X": "mismo punto en California State Plane; redundante",
    "Y": "mismo punto en California State Plane; redundante",
    "data_loaded_at": "sello de carga del portal, sin valor analitico",
    "Permit Type": "reemplazada por tipo_permiso agrupado",
    "Agent": "reemplazada por agente agrupado",
    "supervisor_district": "reemplazada por distrito como categorica",
    "AgentPhone": "dato de contacto, sin valor analitico",
    "24/7 Contact": "dato de contacto, sin valor analitico",
    "cnn": "identificador de segmento vial, sin valor explicativo",
}


class ProyectoError(Exception):
    """Error propio del proyecto: permite distinguirlo de los de las librerias."""


def normalizar_nombre(texto):
    """Convierte un nombre de columna a minusculas, sin espacios ni signos.

    Ejemplo
    -------
    >>> normalizar_nombre('Permit Type ')
    'permit_type'
    """
    if not isinstance(texto, str):
        raise ProyectoError(f"Se esperaba texto y se recibio {type(texto).__name__}.")
    limpio = texto.strip().lower()
    limpio = re.sub(r"[^a-z0-9]+", "_", limpio)   # todo lo que no sea letra o digito -> _
    return limpio.strip("_")


def normalizar_columnas(nombres):
    """Aplica normalizar_nombre a una lista y verifica que no se produzcan duplicados."""
    normalizados = [normalizar_nombre(n) for n in nombres]
    if len(set(normalizados)) != len(normalizados):
        raise ProyectoError("La normalizacion produjo nombres duplicados.")
    return normalizados


def localizar_raiz(inicio=None, marcadores=MARCADORES, niveles=5, respaldo=True):
    """Resuelve la raiz del proyecto. Ver seccion 3 del cuaderno de la Fase 1."""
    actual = Path(inicio or Path.cwd()).resolve()
    for candidata in [actual, *actual.parents][:niveles + 1]:
        if any((candidata / m).exists() for m in marcadores):
            return candidata, "estructurado"
    if respaldo:
        return actual, "plano"
    raise FileNotFoundError(f"No se encontro marcador de repositorio desde {actual}.")


def buscar_archivo(nombre, raiz=None, extra=()):
    """Busca un archivo en las ubicaciones canonicas y junto al cuaderno."""
    base = Path(raiz) if raiz is not None else localizar_raiz()[0]
    candidatos = [
        Path.cwd() / nombre, base / nombre,
        base / "data" / "raw" / nombre, base / "data" / "processed" / nombre,
        base / "docs" / nombre, *[Path(d) / nombre for d in extra],
    ]
    vistos = set()
    for ruta in candidatos:
        clave = str(ruta.resolve())
        if clave in vistos:
            continue
        vistos.add(clave)
        if ruta.exists():
            return ruta.resolve()
    return None


def leer_crudo(ruta):
    """Lee el CSV como texto, sin inferencia de tipos.

    La inferencia automatica es inaceptable en esta fuente: las coordenadas usan coma
    decimal y una parte de las fechas admite dos lecturas. Leer todo como texto obliga
    a declarar cada conversion de forma explicita.
    """
    return pd.read_csv(ruta, dtype=str)


def convertir_tipos(datos):
    """Convierte coordenadas, fechas y centinelas. No elimina filas ni columnas."""
    d = datos.copy()
    for col in COLS_COORDENADA:
        if col in d.columns:
            d[col] = pd.to_numeric(d[col].str.replace(",", ".", regex=False), errors="coerce")
    for col in COLS_FECHA:
        if col in d.columns:
            d[col] = pd.to_datetime(d[col], format=FORMATO_FECHA, errors="coerce")
    if "data_as_of" in d.columns:
        d["data_as_of"] = pd.to_datetime(d["data_as_of"], format=FORMATO_SELLO, errors="coerce")
    if "permit_zipcode" in d.columns:
        d["permit_zipcode"] = d["permit_zipcode"].replace("0", np.nan)   # centinela
    return d


def derivar_variables(datos, n_tipos=12, n_agentes=12):
    """Deriva el objetivo y las variables explicativas de la pregunta.

    Lanza
    -----
    KeyError
        Si falta alguna de las columnas necesarias para derivar.
    """
    requeridas = {"permit_start_date", "permit_end_date", "Approved Date",
                  "Permit Type", "Agent", "supervisor_district", "permit_number"}
    faltan = requeridas - set(datos.columns)
    if faltan:
        raise KeyError(f"Faltan columnas para derivar: {sorted(faltan)}")

    d = datos.copy()
    d["duracion_dias"] = (d["permit_end_date"] - d["permit_start_date"]).dt.days
    d["n_segmentos"] = d.groupby("permit_number")["permit_number"].transform("size")
    d["lag_aprob_inicio"] = (d["permit_start_date"] - d["Approved Date"]).dt.days
    d["anio_aprobacion"] = d["Approved Date"].dt.year
    d["mes_inicio"] = d["permit_start_date"].dt.month

    principales_tipo = d["Permit Type"].value_counts().head(n_tipos).index
    d["tipo_permiso"] = np.where(d["Permit Type"].isin(principales_tipo),
                                 d["Permit Type"], "Otros")
    principales_agente = d["Agent"].value_counts().head(n_agentes).index
    d["agente"] = np.where(d["Agent"].isin(principales_agente),
                           d["Agent"].fillna("Desconocido"), "Otros")
    d["distrito"] = d["supervisor_district"].fillna("Desconocido").astype(str)
    return d


def construir_conjunto_trabajo(datos_crudos):
    """Aplica el plan de saneamiento completo: tipos, derivadas, descartes y filtros.

    Implementa la decision justificada en la seccion 5.2 del cuaderno de la Fase 1.
    """
    d = derivar_variables(convertir_tipos(datos_crudos))
    d = d.drop(columns=[c for c in DESCARTES if c in d.columns])
    d = d.dropna(subset=["duracion_dias"])      # sin objetivo no hay caso analizable
    d = d.drop_duplicates()
    return d.reset_index(drop=True)


def clasificar_rol(serie):
    """Asigna el rol analitico de una variable segun las reglas del curso."""
    s = serie.dropna()
    if s.empty:
        return "vacia"
    unicos = s.nunique()
    proporcion = unicos / len(s)
    es_numerica = pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_bool_dtype(s)
    if pd.api.types.is_datetime64_any_dtype(s):
        return "fecha"
    if unicos == 2:
        return "binaria"
    if es_numerica:
        entera = s.mod(1).eq(0).all()
        if entera and proporcion > 0.95 and len(s) > 50:
            return "identificador"
        if entera and (unicos <= 20 or proporcion < 0.05):
            return "discreta"
        return "continua"
    if pd.to_datetime(s.head(200), errors="coerce", format="mixed").notna().mean() > 0.9:
        return "fecha"
    if s.astype(str).str.len().mean() > 60:
        return "texto libre"
    if proporcion > 0.95 and len(s) > 50:
        return "identificador"
    return "nominal" if unicos <= 15 else "alta cardinalidad"


def contar_roles(datos):
    """Cuenta variables por rol analitico."""
    return pd.Series([clasificar_rol(datos[c]) for c in datos.columns]).value_counts().to_dict()


def perfilar(datos):
    """Devuelve rol, dtype, nulos y cardinalidad por columna."""
    return pd.DataFrame({
        "rol": [clasificar_rol(datos[c]) for c in datos.columns],
        "dtype": datos.dtypes.astype(str),
        "n_nulos": datos.isna().sum(),
        "pct_nulos": (datos.isna().mean() * 100).round(2),
        "n_unicos": datos.nunique(dropna=True),
    }, index=datos.columns)
'''

ARCHIVO_MODULO = DIR_SRC / "utilidades.py"
ARCHIVO_MODULO.write_text(CODIGO_MODULO, encoding="utf-8")
print("Módulo escrito en:", ARCHIVO_MODULO.relative_to(RAIZ).as_posix(),
      f"({ARCHIVO_MODULO.stat().st_size} bytes)")

Módulo escrito en: src/utilidades.py (8183 bytes)


Para importarlo hay que agregar `src/` a la lista de rutas donde Python busca módulos.
`invalidate_caches` y `reload` permiten volver a ejecutar el cuaderno sin reiniciar el kernel:
sin ellos, Python seguiría usando la versión del módulo que cargó la primera vez.

In [10]:
if str(DIR_SRC.resolve()) not in sys.path:
    sys.path.insert(0, str(DIR_SRC.resolve()))

importlib.invalidate_caches()
import utilidades                      # el módulo recién escrito
importlib.reload(utilidades)           # por si el cuaderno se vuelve a ejecutar

print("Importado:", utilidades.__name__, "desde", Path(utilidades.__file__).as_posix())
print(utilidades.normalizar_nombre("Permit Type "))
print(utilidades.normalizar_columnas(["Approved Date", "Cross Street 1", "24/7 Contact"]))
print("Funciones expuestas:", sorted(n for n in dir(utilidades)
                                     if not n.startswith("_")
                                     and callable(getattr(utilidades, n))))

Importado: utilidades desde /Users/ricardo/Desktop/Repositorios/proyecto-grupo5-mcdi500/src/utilidades.py
permit_type
['approved_date', 'cross_street_1', '24_7_contact']
Funciones expuestas: ['Path', 'ProyectoError', 'buscar_archivo', 'clasificar_rol', 'construir_conjunto_trabajo', 'contar_roles', 'convertir_tipos', 'derivar_variables', 'leer_crudo', 'localizar_raiz', 'normalizar_columnas', 'normalizar_nombre', 'perfilar']


### 4.1 Pruebas del módulo (casos normal, límite y excepción)

Tres tipos de caso por cada función relevante. La rúbrica los exige y, además, son lo que
permite cambiar el módulo más adelante sin romper lo que ya funcionaba.

In [16]:
def probar_modulo():
    """Ejecuta las pruebas del modulo: casos normal, limite y excepciones."""
    import tempfile

    # --- Caso normal ---
    assert utilidades.normalizar_nombre("Approved Date") == "approved_date"
    assert utilidades.normalizar_columnas(["A B", "c-d"]) == ["a_b", "c_d"]
    assert utilidades.localizar_raiz(RAIZ)[0] == RAIZ
    print("[OK] Caso normal: normalización y localización de la raíz")

    # --- Caso limite: signos consecutivos, lista vacia y busqueda sin coincidencia ---
    assert utilidades.normalizar_nombre("  ¿24/7 Contact?  ") == "24_7_contact"
    assert utilidades.normalizar_columnas([]) == []
    assert utilidades.buscar_archivo("no_existe_jamas_12345.csv", RAIZ) is None
    print("[OK] Caso límite: signos, lista vacía y archivo inexistente")

    # --- Caso limite: carpeta sin marcador de repositorio ---
    with tempfile.TemporaryDirectory() as temporal:
        ruta, modo = utilidades.localizar_raiz(temporal, niveles=0)
        assert Path(ruta) == Path(temporal).resolve() and modo == "plano"
        (Path(temporal) / ".gitignore").touch()
        assert utilidades.localizar_raiz(temporal, niveles=0)[1] == "estructurado"
        print("[OK] Caso límite: respaldo sin marcador y detección con marcador")

        # --- Excepcion 1: sin marcador y sin respaldo ---
        (Path(temporal) / ".gitignore").unlink()
        try:
            utilidades.localizar_raiz(temporal, niveles=0, respaldo=False)
            raise AssertionError("debio lanzar FileNotFoundError")
        except FileNotFoundError as e:
            print(f"[OK] Excepción capturada (raíz): {str(e).split('.')[0]}")

    # --- Excepcion 2: tipo incorrecto ---
    try:
        utilidades.normalizar_nombre(42)
        raise AssertionError("debio lanzar ProyectoError")
    except utilidades.ProyectoError as e:
        print(f"[OK] Excepción capturada (tipo): {e}")

    # --- Excepcion 3: la normalizacion colapsa dos nombres en uno ---
    try:
        utilidades.normalizar_columnas(["Agent", " agent "])
        raise AssertionError("debio lanzar ProyectoError")
    except utilidades.ProyectoError as e:
        print(f"[OK] Excepción capturada (duplicados): {e}")

    # --- Excepcion 4: faltan columnas para derivar ---
    try:
        utilidades.derivar_variables(pd.DataFrame({"permit_number": [1]}))
        raise AssertionError("debio lanzar KeyError")
    except KeyError as e:
        print(f"[OK] Excepción capturada (columnas): {str(e)[:60]}...")

    print("\n[OK] 8/8 pruebas del módulo superadas")
    return True


_, ANEXO_G1 = capturar(probar_modulo)      # Anexo G del informe (Fase 1)

[OK] Caso normal: normalización y localización de la raíz
[OK] Caso límite: signos, lista vacía y archivo inexistente
[OK] Caso límite: respaldo sin marcador y detección con marcador
[OK] Excepción capturada (raíz): No se encontro marcador de repositorio desde /private/var/folders/25/1d9z1__s1fl2f9r35bnxb_080000gn/T/tmps99wen0r
[OK] Excepción capturada (tipo): Se esperaba texto y se recibio int.
[OK] Excepción capturada (duplicados): La normalizacion produjo nombres duplicados.
[OK] Excepción capturada (columnas): "Faltan columnas para derivar: ['Agent', 'Approved Date', 'P...

[OK] 8/8 pruebas del módulo superadas


> **Por qué una excepción propia.** `ProyectoError` hereda de `Exception` y permite
> distinguir un fallo del proyecto de uno de pandas o de Python. Cuando el cuaderno falle
> dentro de seis meses, la diferencia entre «se rompió algo» y «el módulo detectó que la
> normalización iba a perder una columna» es lo que separa diez minutos de una tarde.

## 5. Selección y ficha del conjunto de datos

### 5.1 Procedencia

Los metadatos provienen del portal oficial, no de la descripción de la descarga.

**Cita en APA 7**, con fecha de recuperación porque la fuente cambia a diario:

> San Francisco Public Works. (2026). *Active street-use permits* [Conjunto de datos].
> DataSF. Recuperado el 9 de septiembre de 2026, de
> https://data.sf.gov/City-Infrastructure/Active-Street-Use-Permits/x8nh-xzn6/about_data

El portal declara además el criterio de inclusión, que resulta determinante para interpretar
los resultados: se publican los permisos con estado activo o aprobado y, **solo para los tipos
Excavation y TempOcc**, aquellos que no han superado su fecha de término.

In [11]:
FICHA = {
    "titulo": "Active Street-Use Permits",
    "identificador": "x8nh-xzn6",
    "inventario": "DPW-0022",
    "autor": "San Francisco Public Works",
    "plataforma": "DataSF",
    "url": "https://data.sf.gov/City-Infrastructure/Active-Street-Use-Permits/"
           "x8nh-xzn6/about_data",
    "licencia": "Open Data Commons Public Domain Dedication and License (PDDL) 1.0",
    "licencia_abierta": True,
    "publicado": "2025-01-17",
    "corte_declarado": "2026-09-09",     # se verifica contra data_as_of en la seccion 5.4
    "frecuencia": "Diaria",
    "descargado": "2026-09-09",
    "archivo": "Active_Street-Use_Permits_20260909.csv",
    "criterio_inclusion": (
        "Permisos con estado activo o aprobado. Para los tipos Excavation y TempOcc, "
        "unicamente aquellos que no han superado su fecha de termino."
    ),
    "historico": "Street-Use Permits (b6tj-gt35), sin filtro de vigencia",
    # Estructura del archivo TAL COMO SE DESCARGA, con los tipos ya convertidos.
    "filas": 6852,
    "columnas": 29,
    "roles": {"alta cardinalidad": 18, "fecha": 5, "nominal": 2, "continua": 2,
              "texto libre": 1, "binaria": 1, "discreta": 0, "ordinal": 0},
    "pct_nulos_max_variable": 96.44,     # CurbRampWork
    "pct_nulos_min_no_cero": 0.03,
}

for clave in ("titulo", "autor", "plataforma", "licencia", "corte_declarado",
              "frecuencia", "archivo"):
    print(f"  {clave:20}: {FICHA[clave]}")
print(f"  {'dimensiones':20}: {FICHA['filas']} filas x {FICHA['columnas']} columnas")

  titulo              : Active Street-Use Permits
  autor               : San Francisco Public Works
  plataforma          : DataSF
  licencia            : Open Data Commons Public Domain Dedication and License (PDDL) 1.0
  corte_declarado     : 2026-09-09
  frecuencia          : Diaria
  archivo             : Active_Street-Use_Permits_20260909.csv
  dimensiones         : 6852 filas x 29 columnas


### 5.2 Evaluación contra los criterios del curso

El curso exige que el conjunto reúna una variedad mínima de roles analíticos y una presencia
de faltantes que justifique el preprocesamiento. Se evalúa primero el archivo **tal como se
descarga**.

In [12]:
CRITERIOS = {
    "filas": ("Filas", 2000, "Permite agrupar por categoria sin grupos vacios"),
    "columnas": ("Columnas", 12, "Asegura variedad de roles analiticos"),
    "continua": ("Numericas continuas", 2, "Escalamiento y valores atipicos"),
    "discreta": ("Numerica discreta", 1, "Obliga a decidir numero frente a categoria"),
    "nominal": ("Categoricas nominales", 2, "Codificacion One-Hot"),
    "ordinal_o_binaria": ("Binaria u ordinal", 1, "Codificacion con orden declarado"),
    "fecha": ("Fecha", 1, "Parseo y derivacion de variables temporales"),
    "alta_cardinalidad": ("Texto o alta cardinalidad", 1, "Normalizacion y agrupacion"),
}


def evaluar_criterios(ficha, criterios=CRITERIOS):
    """
    Contrasta una ficha de conjunto de datos con los minimos exigidos por el curso.

    Nota sobre el criterio de faltantes: el enunciado dice "alguna variable con >=1% de
    nulos", de modo que basta con que exista una. Se evalua por tanto contra el maximo
    observado y no contra el minimo, que exigiria que TODAS lo superaran.

    Lanza
    -----
    KeyError
        Si la ficha no declara el conteo de variables por rol.
    """
    if "roles" not in ficha:
        raise KeyError("La ficha debe declarar el conteo de variables por rol analitico.")
    r = ficha["roles"]
    obs = {
        "filas": ficha.get("filas", 0), "columnas": ficha.get("columnas", 0),
        "continua": r.get("continua", 0), "discreta": r.get("discreta", 0),
        "nominal": r.get("nominal", 0),
        "ordinal_o_binaria": r.get("ordinal", 0) + r.get("binaria", 0),
        "fecha": r.get("fecha", 0),
        "alta_cardinalidad": r.get("alta cardinalidad", 0) + r.get("texto libre", 0),
    }
    filas = [{"criterio": e, "minimo": m, "observado": obs[k],
              "cumple": "si" if obs[k] >= m else "NO", "por que se pide": motivo}
             for k, (e, m, motivo) in criterios.items()]
    mx = ficha.get("pct_nulos_max_variable", 0.0)
    filas.append({"criterio": "Alguna variable con >=1% de nulos", "minimo": 1.0,
                  "observado": mx, "cumple": "si" if mx >= 1.0 else "NO",
                  "por que se pide": "Un conjunto perfecto no tiene preprocesamiento que justificar"})
    filas.append({"criterio": "Ninguna variable sobre 60% de nulos", "minimo": 60.0,
                  "observado": mx, "cumple": "si" if mx <= 60.0 else "NO",
                  "por que se pide": "Por encima de eso la variable no es recuperable"})
    tabla = pd.DataFrame(filas)
    for col in ("minimo", "observado"):
        tabla[col] = tabla[col].map(lambda v: f"{v:g}")
    return tabla


evaluacion_archivo = evaluar_criterios(FICHA)
evaluacion_archivo

,criterio,minimo,observado,cumple,por que se pide
0,Filas,2000,6852,si,Permite agrupar por categoria sin grupos vacios
1,Columnas,12,29,si,Asegura variedad de roles analiticos
2,Numericas continuas,2,2,si,Escalamiento y valores atipicos
3,Numerica discreta,1,0,NO,Obliga a decidir numero frente a categoria
4,Categoricas nominales,2,2,si,Codificacion One-Hot
5,Binaria u ordinal,1,1,si,Codificacion con orden declarado
6,Fecha,1,5,si,Parseo y derivacion de variables temporales
7,Texto o alta cardinalidad,1,19,si,Normalizacion y agrupacion
8,Alguna variable con >=1% de nulos,1,96.44,si,Un conjunto perfecto no tiene preprocesamiento...
9,Ninguna variable sobre 60% de nulos,60,96.44,NO,Por encima de eso la variable no es recuperable


In [13]:
incumplidos_archivo = evaluacion_archivo.loc[
    evaluacion_archivo["cumple"] == "NO", "criterio"].tolist()
print(f"Archivo tal como se descarga: "
      f"{len(evaluacion_archivo) - len(incumplidos_archivo)}/{len(evaluacion_archivo)} criterios")
for x in incumplidos_archivo:
    print("  NO cumple:", x)

Archivo tal como se descarga: 8/10 criterios
  NO cumple: Numerica discreta
  NO cumple: Ninguna variable sobre 60% de nulos


> **Léase con atención el resultado.** El archivo **no cumple** dos criterios: no hay
> ninguna variable numérica discreta, y tres columnas superan el 60 % de valores faltantes
> —`CurbRampWork` con 96,4 %, `bpa` con 95,7 % y `permit_address` con 63,1 %—. Descubrirlo
> aquí, y no en la Fase 2, es exactamente para lo que sirve evaluar el conjunto antes de
> comprometerse con él.
>
> **El archivo tal como se descarga no es el conjunto de análisis.** Tres grupos de decisiones
> lo separan del conjunto de trabajo:
>
> 1. **Conversión de tipos, nunca por inferencia.** Las coordenadas usan coma decimal y
>    ninguna se interpreta como número al leer el archivo. En las fechas, una parte de los
>    registros admite dos lecturas. La sección 5.4 lo mide.
> 2. **Catorce columnas descartadas**, por cuatro motivos: exceso de faltantes, varianza nula,
>    redundancia geométrica y ausencia de valor explicativo. El motivo de cada una está
>    declarado en `utilidades.DESCARTES`.
> 3. **Filtro de filas.** Solo se conservan los permisos con ambas fechas presentes, porque
>    sin ellas la duración autorizada —la variable objetivo— no existe. Y se eliminan las
>    filas exactamente duplicadas que quedan tras los descartes.
>
> *Enfoque descartado:* imputar las fechas ausentes para conservar los 6.852 registros.
> Imputar la variable objetivo fabrica los casos que después se pretende explicar.
>
> Las derivaciones del plan aportan además las variables discretas que faltaban: la duración
> en días, el número de segmentos por permiso, el retraso entre aprobación e inicio, el año de
> aprobación y el mes de inicio.

### 5.3 Diccionario de variables

El diccionario declara dos atributos por variable. El **rol analítico** determina el
preprocesamiento de la Fase 2. El **uso** determina si la variable participa en el análisis y
en qué papel, y es el mecanismo que impide la fuga de información: se decide una vez aquí y
las fases siguientes lo leen sin volver a decidirlo.

In [14]:
ARCHIVO = utilidades.buscar_archivo(FICHA["archivo"], RAIZ)

if ARCHIVO is None:
    crudo = trabajo = None
    print("[PENDIENTE] No se encontró", FICHA["archivo"])
    print("Descárguelo desde:", FICHA["url"])
    print("y guárdelo en    :", DIR_CRUDO.relative_to(RAIZ).as_posix())
else:
    crudo = utilidades.leer_crudo(ARCHIVO)
    trabajo = utilidades.construir_conjunto_trabajo(crudo)

    conv = utilidades.convertir_tipos(crudo)
    sin_objetivo = int((conv["permit_end_date"] - conv["permit_start_date"]).dt.days.isna().sum())
    duplicadas = crudo.shape[0] - sin_objetivo - trabajo.shape[0]

    print("Archivo            :", ARCHIVO.relative_to(RAIZ).as_posix())
    print("Crudo              :", crudo.shape)
    print("Sin ambas fechas   :", sin_objetivo, "registros descartados")
    print("Duplicados exactos :", duplicadas, "registros descartados")
    print("Conjunto de trabajo:", trabajo.shape)
    print("Permisos únicos    :", trabajo.permit_number.nunique())
    print("Columnas descartadas:", len(utilidades.DESCARTES))

FICHA_GRUPO = {
    "titulo": FICHA["titulo"] + " — conjunto de trabajo",
    "autor": FICHA["autor"],
    "plataforma": FICHA["plataforma"],
    "url": FICHA["url"],
    "licencia_abierta": FICHA["licencia_abierta"],
    "archivo": FICHA["archivo"],
    "filas": 6010,
    "columnas": 23,
    "roles": {"alta cardinalidad": 7, "discreta": 5, "fecha": 4, "nominal": 3,
              "continua": 2, "texto libre": 1, "binaria": 1, "ordinal": 0},
    "pct_nulos_max_variable": 38.47,     # Inspector
    "pct_nulos_min_no_cero": 0.05,       # analysis_neighborhood
}

evaluacion = evaluar_criterios(FICHA_GRUPO)
incumplidos = evaluacion.loc[evaluacion["cumple"] == "NO", "criterio"].tolist()

if trabajo is not None:
    # La ficha es una declaracion; aqui se comprueba contra el conjunto real.
    nulos = (trabajo.isna().mean() * 100).round(2)
    assert FICHA_GRUPO["filas"] == len(trabajo), "La ficha declara otro numero de filas."
    assert FICHA_GRUPO["columnas"] == trabajo.shape[1], "La ficha declara otro numero de columnas."
    assert abs(FICHA_GRUPO["pct_nulos_max_variable"] - nulos.max()) < 0.01
    print("\n[OK] La ficha declarada coincide con el conjunto de trabajo real")

print(f"\nConjunto de trabajo: {len(evaluacion) - len(incumplidos)}/{len(evaluacion)} criterios")
print("Incumplidos:", incumplidos or "ninguno")
evaluacion

Archivo            : data/raw/Active_Street-Use_Permits_20260909.csv
Crudo              : (6852, 29)
Sin ambas fechas   : 838 registros descartados
Duplicados exactos : 4 registros descartados
Conjunto de trabajo: (6010, 23)
Permisos únicos    : 2139
Columnas descartadas: 14

[OK] La ficha declarada coincide con el conjunto de trabajo real

Conjunto de trabajo: 10/10 criterios
Incumplidos: ninguno


,criterio,minimo,observado,cumple,por que se pide
0,Filas,2000,6010,si,Permite agrupar por categoria sin grupos vacios
1,Columnas,12,23,si,Asegura variedad de roles analiticos
2,Numericas continuas,2,2,si,Escalamiento y valores atipicos
3,Numerica discreta,1,5,si,Obliga a decidir numero frente a categoria
4,Categoricas nominales,2,3,si,Codificacion One-Hot
5,Binaria u ordinal,1,1,si,Codificacion con orden declarado
6,Fecha,1,4,si,Parseo y derivacion de variables temporales
7,Texto o alta cardinalidad,1,8,si,Normalizacion y agrupacion
8,Alguna variable con >=1% de nulos,1,38.47,si,Un conjunto perfecto no tiene preprocesamiento...
9,Ninguna variable sobre 60% de nulos,60,38.47,si,Por encima de eso la variable no es recuperable


In [15]:
DICCIONARIO = [
    # (variable, rol analitico, uso, descripcion)
    ("duracion_dias",         "discreta",          "objetivo",        "Duracion autorizada en dias: permit_end_date - permit_start_date"),
    ("tipo_permiso",          "nominal",           "explicativa",     "Tipo de permiso; 12 niveles principales + Otros"),
    ("analysis_neighborhood", "alta cardinalidad", "explicativa",     "Barrio de emplazamiento; 41 niveles"),
    ("agente",                "nominal",           "explicativa",     "Agente solicitante; 12 principales + Otros. NO es el organismo emisor"),
    ("anio_aprobacion",       "discreta",          "explicativa",     "Anio de aprobacion, derivado de Approved Date"),
    ("distrito",              "nominal",           "explicativa",     "Distrito del supervisor; 11 niveles + Desconocido"),
    ("n_segmentos",           "discreta",          "explicativa",     "Segmentos de calle que abarca el permiso; proxy del tamanio del proyecto"),
    ("lag_aprob_inicio",      "discreta",          "explicativa",     "Dias entre aprobacion e inicio; negativo en aprobaciones retroactivas"),
    ("mes_inicio",            "discreta",          "explicativa",     "Mes de inicio; estacionalidad"),
    ("Permit Purpose",        "texto libre",       "explicativa",     "Descripcion del proposito; 26,3% ausente y la ausencia es informativa"),
    ("Inspector",             "alta cardinalidad", "explicativa",     "Inspector municipal asignado; 38,5% ausente y la ausencia es informativa"),
    ("permit_zipcode",        "alta cardinalidad", "explicativa",     "Codigo postal; el valor 0 es centinela de ausencia"),
    ("Status",                "binaria",           "no_usada",        "Estado administrativo; no es un factor de la pregunta"),
    ("permit_number",         "alta cardinalidad", "grupo",           "Identificador del permiso; unidad de agrupacion para la particion"),
    ("permit_start_date",     "fecha",             "fuente_objetivo", "Componente del objetivo; no puede ser explicativa"),
    ("permit_end_date",       "fecha",             "fuente_objetivo", "Componente del objetivo; no puede ser explicativa"),
    ("Approved Date",         "fecha",             "fuente_derivada", "Origen de anio_aprobacion y lag_aprob_inicio"),
    ("data_as_of",            "fecha",             "trazabilidad",    "Sello de actualizacion del portal"),
    ("Latitude",              "continua",          "validacion",      "Latitud; validacion espacial y mapas de la Fase 4"),
    ("Longitude",             "continua",          "validacion",      "Longitud; validacion espacial y mapas de la Fase 4"),
    ("streetname",            "alta cardinalidad", "contexto",        "Calle principal del segmento"),
    ("Cross Street 1",        "alta cardinalidad", "contexto",        "Primera calle transversal"),
    ("Cross Street 2",        "alta cardinalidad", "contexto",        "Segunda calle transversal; 16,0% ausente"),
]

diccionario = pd.DataFrame(DICCIONARIO,
                           columns=["variable", "rol_analitico", "uso", "descripcion"])

OBJETIVO     = diccionario.loc[diccionario.uso == "objetivo", "variable"].item()
EXPLICATIVAS = diccionario.loc[diccionario.uso == "explicativa", "variable"].tolist()
GRUPO_VAR    = diccionario.loc[diccionario.uso == "grupo", "variable"].item()
PROHIBIDAS   = diccionario.loc[diccionario.uso.isin(["fuente_objetivo", "no_usada"]),
                               "variable"].tolist()

# Verificacion: el diccionario debe cubrir exactamente las columnas declaradas en la ficha.
assert len(diccionario) == FICHA_GRUPO["columnas"], "El diccionario no coincide con la ficha."
assert OBJETIVO not in EXPLICATIVAS
assert not set(PROHIBIDAS) & set(EXPLICATIVAS)
if trabajo is not None:
    assert set(diccionario.variable) == set(trabajo.columns), "Discrepancia de nombres."

print(diccionario["rol_analitico"].value_counts().to_string())
print(f"\nObjetivo: {OBJETIVO} | explicativas: {len(EXPLICATIVAS)} | "
      f"grupo: {GRUPO_VAR} | prohibidas: {PROHIBIDAS}")
diccionario

rol_analitico
alta cardinalidad    7
discreta             5
fecha                4
nominal              3
continua             2
texto libre          1
binaria              1

Objetivo: duracion_dias | explicativas: 11 | grupo: permit_number | prohibidas: ['Status', 'permit_start_date', 'permit_end_date']


,variable,rol_analitico,uso,descripcion
0,duracion_dias,discreta,objetivo,Duracion autorizada en dias: permit_end_date -...
1,tipo_permiso,nominal,explicativa,Tipo de permiso; 12 niveles principales + Otros
2,analysis_neighborhood,alta cardinalidad,explicativa,Barrio de emplazamiento; 41 niveles
3,agente,nominal,explicativa,Agente solicitante; 12 principales + Otros. NO...
4,anio_aprobacion,discreta,explicativa,"Anio de aprobacion, derivado de Approved Date"
5,distrito,nominal,explicativa,Distrito del supervisor; 11 niveles + Desconocido
6,n_segmentos,discreta,explicativa,Segmentos de calle que abarca el permiso; prox...
7,lag_aprob_inicio,discreta,explicativa,Dias entre aprobacion e inicio; negativo en ap...
8,mes_inicio,discreta,explicativa,Mes de inicio; estacionalidad
9,Permit Purpose,texto libre,explicativa,"Descripcion del proposito; 26,3% ausente y la ..."


> **El rol no basta: hace falta el uso.** `permit_start_date` y `permit_end_date` tienen un
> rol analítico impecable —son fechas— y sin embargo **no pueden usarse para explicar nada**,
> porque la duración es su resta. Usarlas equivaldría a predecir una resta a partir de sus dos
> sumandos. Esa distinción no cabe en la columna de rol, y por eso el diccionario declara una
> columna adicional.
>
> Dos declaraciones más que respaldamos. `permit_number` es la **variable de
> agrupación**, no un identificador descartable: un permiso abarca hasta 205 segmentos de calle
> que comparten agente, propósito y ambas fechas, de modo que la partición de la Fase 2 debe
> agrupar por él. Y `Latitude` y `Longitude` **no son explicativas**: la ubicación territorial
> entra por `analysis_neighborhood`, que es interpretable en la respuesta; las coordenadas se
> reservan para validar la consistencia del barrio.

### 5.4 Comprobación del archivo y verificaciones de lectura

Se contrasta la estructura real del archivo con la declarada y se verifican, con cifras, las
tres afirmaciones sobre las que descansa el plan de saneamiento.

In [17]:
if crudo is not None:
    CORTE = conv["data_as_of"].max()
    print("Dimensiones reales :", crudo.shape)
    print("Declarado en ficha :", (FICHA["filas"], FICHA["columnas"]))
    print("Coincidencia       :",
          "[OK]" if crudo.shape == (FICHA["filas"], FICHA["columnas"]) else "[REVISAR]")
    print("Duplicados exactos en el crudo:", int(crudo.duplicated().sum()))
    print("Duplicados por (timestamp, permiso):",
          int(crudo.duplicated(subset=["permit_number", "cnn"]).sum()))
    print("\nCorte observado en data_as_of :", f"{CORTE:%Y-%m-%d %H:%M}")
    print("Corte declarado en la ficha   :", FICHA["corte_declarado"])
    print("Coincidencia                  :",
          "[OK]" if str(CORTE.date()) == FICHA["corte_declarado"]
          else "[REVISAR] actualice corte_declarado en FICHA")
    print("\nPerfil del conjunto de trabajo:")
    display(utilidades.perfilar(trabajo))
else:
    print("[PENDIENTE] Sin archivo no hay comprobación posible.")

Dimensiones reales : (6852, 29)
Declarado en ficha : (6852, 29)
Coincidencia       : [OK]
Duplicados exactos en el crudo: 0
Duplicados por (timestamp, permiso): 0

Corte observado en data_as_of : 2026-09-09 03:15
Corte declarado en la ficha   : 2026-09-09
Coincidencia                  : [OK]

Perfil del conjunto de trabajo:


,rol,dtype,n_nulos,pct_nulos,n_unicos
permit_number,alta cardinalidad,object,0,0.00,2139
streetname,alta cardinalidad,object,0,0.00,873
Cross Street 1,alta cardinalidad,object,0,0.00,1352
Cross Street 2,alta cardinalidad,object,961,15.99,1276
Permit Purpose,texto libre,object,1581,26.31,887
Approved Date,fecha,datetime64[ns],4,0.07,645
Status,binaria,object,0,0.00,2
permit_zipcode,alta cardinalidad,object,817,13.59,29
permit_start_date,fecha,datetime64[ns],0,0.00,685
permit_end_date,fecha,datetime64[ns],0,0.00,217


**Verificación 1 · las coordenadas usan coma decimal.**

In [19]:
#TODO:Código verificación

**Verificación 2 · las fechas admiten dos lecturas.** Conviene separar dos preguntas que
suelen confundirse: si el formato es consistente, y cuántos registros son realmente ambiguos.

In [20]:
#TODO: Código consistencia

In [21]:
#TODO: Código ambiguedad

**Verificación 3 · el sesgo de selección de la fuente.** El portal publica solo permisos
vigentes, de modo que un permiso corto iniciado hace años ya salió del registro y uno largo
sigue dentro. Si el mecanismo opera, la duración **mínima** observada debe crecer con la
antigüedad del inicio.

In [22]:
#TODO: Código antiguedad

**Verificación 4 · la ausencia es informativa.** Antes de decidir cómo tratar los valores
faltantes hay que saber si su ausencia significa algo. Imputar una ausencia informativa borra
la información que contiene.

In [23]:
#TODO: Código Ausencia